# Notebook 6 – Interaction Features

## Learning Objectives
Understand interaction features; create multiplication, addition, subtraction, division, polynomial, pairwise, and domain-based interactions; explain when they help and when they add complexity.

## 1. What are Interaction Features?
An interaction feature combines two or more existing features to represent a relationship between them.

**Example:** In retail, quantity multiplied by unit price represents line-item value.

## 2. Sample Dataset

In [1]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    "customer_id":[1,2,3,4,5],
    "age":[22,30,42,35,50],
    "annual_income":[300000,500000,900000,650000,1200000],
    "experience_years":[1,5,18,10,25],
    "quantity":[2,1,5,3,4],
    "unit_price":[100,250,80,150,300],
    "total_spending":[5000,7000,12000,9000,20000],
    "number_of_orders":[2,4,8,3,10]
})
data

,customer_id,age,annual_income,experience_years,quantity,unit_price,total_spending,number_of_orders
0,1,22,300000,1,2,100,5000,2
1,2,30,500000,5,1,250,7000,4
2,3,42,900000,18,5,80,12000,8
3,4,35,650000,10,3,150,9000,3
4,5,50,1200000,25,4,300,20000,10


**Output Explanation:** Each row represents a customer with demographic, income, experience, and purchasing information.

## 3. Feature × Feature (Multiplication)
Multiplication can represent combined scale or total quantity.

**Example:** Quantity multiplied by price.

In [2]:
data["income_age_interaction"] = data["annual_income"] * data["age"]
data["quantity_price_interaction"] = data["quantity"] * data["unit_price"]
data[["annual_income","age","income_age_interaction","quantity","unit_price","quantity_price_interaction"]]

,annual_income,age,income_age_interaction,quantity,unit_price,quantity_price_interaction
0,300000,22,6600000,2,100,200
1,500000,30,15000000,1,250,250
2,900000,42,37800000,5,80,400
3,650000,35,22750000,3,150,450
4,1200000,50,60000000,4,300,1200


**Output Explanation:** Quantity-price interaction represents item value. Income-age is a mathematical combination and needs a domain reason before retention.

## 4. Feature + Feature (Addition)
Addition can combine values that contribute to a meaningful total. A mathematical sum is not automatically a meaningful business feature.

In [3]:
data["age_plus_experience"] = data["age"] + data["experience_years"]
data[["age","experience_years","age_plus_experience"]]

,age,experience_years,age_plus_experience
0,22,1,23
1,30,5,35
2,42,18,60
3,35,10,45
4,50,25,75


**Output Explanation:** This is only a numeric sum. Age and experience are different concepts, so it may not be useful without a clear interpretation.

## 5. Feature − Feature (Subtraction)
Subtraction can represent a gap between related measures.

In [4]:
data["age_experience_gap"] = data["age"] - data["experience_years"]
data[["age","experience_years","age_experience_gap"]]

,age,experience_years,age_experience_gap
0,22,1,21
1,30,5,25
2,42,18,24
3,35,10,25
4,50,25,25


**Output Explanation:** The difference may approximate years before the start of recorded experience, depending on the definitions of the columns.

## 6. Feature / Feature (Division)
Division creates per-unit or relative features. Protect against zero denominators.

In [5]:
data["spending_per_order"] = data["total_spending"] / data["number_of_orders"].replace(0,np.nan)
data["experience_to_age_ratio"] = data["experience_years"] / data["age"].replace(0,np.nan)
data[["spending_per_order","experience_to_age_ratio"]]

,spending_per_order,experience_to_age_ratio
0,2500.0,0.045455
1,1750.0,0.166667
2,1500.0,0.428571
3,3000.0,0.285714
4,2000.0,0.500000


**Output Explanation:** Spending per order is average spending per recorded order. The ratio expresses experience relative to age.

## 7. Polynomial Features
Polynomial features create powers and combinations such as x² and x₁x₂. They can represent non-linear relationships.

In [6]:
from sklearn.preprocessing import PolynomialFeatures

base = data[["age","experience_years"]]
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_array = poly.fit_transform(base)
poly_df = pd.DataFrame(poly_array, columns=poly.get_feature_names_out(base.columns))
poly_df

,age,experience_years,age^2,age experience_years,experience_years^2
0,22.0,1.0,484.0,22.0,1.0
1,30.0,5.0,900.0,150.0,25.0
2,42.0,18.0,1764.0,756.0,324.0
3,35.0,10.0,1225.0,350.0,100.0
4,50.0,25.0,2500.0,1250.0,625.0


**Output Explanation:** The output includes original values, squared terms, and a pairwise product. Polynomial features can rapidly increase dataset width and overfitting risk.

## 8. Pairwise Interactions
Pairwise interactions combine selected pairs of features.

In [7]:
selected = data[["age","annual_income","experience_years"]]
generator = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
values = generator.fit_transform(selected)
interaction_df = pd.DataFrame(values, columns=generator.get_feature_names_out(selected.columns))
interaction_df.head()

,age,annual_income,experience_years,age annual_income,age experience_years,annual_income experience_years
0,22.0,300000.0,1.0,6600000.0,22.0,300000.0
1,30.0,500000.0,5.0,15000000.0,150.0,2500000.0
2,42.0,900000.0,18.0,37800000.0,756.0,16200000.0
3,35.0,650000.0,10.0,22750000.0,350.0,6500000.0
4,50.0,1200000.0,25.0,60000000.0,1250.0,30000000.0


**Output Explanation:** The output contains original values and pairwise products. Choose combinations based on the problem rather than generating every possible interaction.

## 9. Domain-Based Interactions
Domain knowledge suggests which columns should be combined.

Examples:
- Retail: quantity × price → line-item sales value.
- Customer analytics: total spending / number of orders → average order value.
- Workforce analytics: experience / age → experience relative to age.
- Banking: debt relative to income → debt burden.

In [8]:
data["line_item_value"] = data["quantity"] * data["unit_price"]
data["average_order_value"] = data["total_spending"] / data["number_of_orders"].replace(0,np.nan)
data[["quantity","unit_price","line_item_value","total_spending","number_of_orders","average_order_value"]]

,quantity,unit_price,line_item_value,total_spending,number_of_orders,average_order_value
0,2,100,200,5000,2,2500.0
1,1,250,250,7000,4,1750.0
2,5,80,400,12000,8,1500.0
3,3,150,450,9000,3,3000.0
4,4,300,1200,20000,10,2000.0


**Output Explanation:** These features have direct retail/customer interpretations. A feature should represent useful information, not just an arbitrary mathematical combination.

## 10. When are Interaction Features Useful?
- The effect of one variable may depend on another.
- A business measure naturally combines multiple columns.
- A simple model needs an explicit representation of a relationship.
- Domain knowledge suggests a meaningful combined effect.

## 11. When Can They Create Unnecessary Complexity?
- Too many combinations create a wide dataset.
- Features are redundant or highly correlated.
- Interactions are difficult to interpret.
- They amplify noise or overfitting.
- There is no business or statistical reason for the combination.